In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

from sensingpy import reader, enums, plot, masks
from sensingpy import bathymetry, preprocessing
from glob import glob

In [ ]:
gee = reader.open(r"E:\GEE_Analysis\GEE\deglint__FORMOSA\2018_10_07.tif")
gee_sen2cor = reader.open(r"E:\GEE_Analysis\GEE__L2_Harmonized\FORMOSA\2018_10_07.tif")
local = reader.open(r"E:\GEE_Analysis\LOCAL\deglint__FORMOSA\S2A_MSI_2018_10_07_11_31_05_T29SPA_L2W.nc").rename_by_enum(enums.SENTINEL2_BANDS)

in_situ = reader.open(r"E:\batimetria\algarve\in_situ\scaled\Armona\201810_10M.tif")
DEPTH = 'depth'
in_situ.rename({'Band 1' : DEPTH})
in_situ.mask(in_situ.select(DEPTH) != -99999.0)
in_situ.mask(in_situ.select(DEPTH) <= 2)
in_situ.add_band(DEPTH, -in_situ.select(DEPTH))

for band in gee.band_names:
    values = gee[band]
    values[values == -np.inf] = np.nan
    gee[band] = values


gee.dropna()
gee_sen2cor.align(gee)
local.align(gee)
in_situ.align(gee)

land_mask = (~np.isnan(local.values)).all(axis=0)
gee.mask(land_mask)
gee_sen2cor.mask(land_mask)

gee['pSDB Green'] = preprocessing.outliers.IQR( bathymetry.models.stumpf_pseudomodel( gee['Rrs_B2'], gee['Rrs_B3']) )
gee_sen2cor['pSDB Green'] = preprocessing.outliers.IQR( bathymetry.models.stumpf_pseudomodel( gee_sen2cor['B2'] / np.pi, gee_sen2cor['B3'] / np.pi) )
local['pSDB Green'] = preprocessing.outliers.IQR( bathymetry.models.stumpf_pseudomodel( local['Rrs_B2'], local['Rrs_B3']) )


df = pd.read_csv(r"E:\batimetria\algarve\results\local\IQR_1.5_pleamar\CSV\calibration_pGreeen_20181007.csv")

models = []
for image in [local, gee, gee_sen2cor]:
    depth = df['in_situ']
    p_green = image.extract_values(df['lon'], df['lat'], bands=['pSDB Green'])

    no_nans = masks.is_valid(depth) & masks.is_valid(p_green)

    model = bathymetry.models.LinearModel().fit(p_green[no_nans], depth[no_nans])
    image['SDB Green'] = model.predict(image['pSDB Green'])

    models.append(
        [model, p_green[no_nans], depth[no_nans]]
    )

In [ ]:
fig, axs = plt.subplots(3, 3, figsize = (18, 18))
cal_plot =  bathymetry.plot.CalibrationPlot(legend_font_size=12)
val_plot = bathymetry.plot.ValidationPlot(legend_font_size=12)

# Parámetros de posición para cada rectángulo: [x, y, width, height]
rect_params = [
    [0.05, 0.65, 0.9, 0.31],  # Fila 1 (Local ACOLITE)
    [0.05, 0.35, 0.9, 0.30],  # Fila 2 (GEE ACOLITE)
    [0.05, 0.05, 0.9, 0.30],  # Fila 3 (GEE Sen2Cor)
]

# Parámetros de posición para los nombres: [x, y]
text_params = [
    [0.02, 0.81],  # Fila 1
    [0.02, 0.48],  # Fila 2
    [0.02, 0.15],  # Fila 3
]

for idx, (image, name) in enumerate(zip([local, gee, gee_sen2cor], ['Local ACOLITE-Fixed', 'GEE + ACOLITE-Fixed', 'GEE Sen2Cor'])):
    depths = in_situ.values[0].ravel()
    sdb_green = image['SDB Green'].ravel()
    no_nans = masks.is_valid(depths) & masks.is_valid(sdb_green)

    error = bathymetry.metrics.ValidationSummary(sdb_green[no_nans], depths[no_nans])

    cal_plot.add_calibration_scatter(models[idx][0], models[idx][1], models[idx][2], axs[idx][0])

    axs[idx, 0].set_xlim(0.9, 1.22)
    axs[idx, 0].set_ylim(0, 10)

    cal_plot.add_legend(axs[idx, 0])
    cal_plot.add_labels(axs[idx, 0], 'Calibration', 'pSDB Green (DII)', 'In Situ (m)')
    val_plot.add_densed_scatter(error, axs[idx, 1], x_min=-4, x_max=25, step=4, density={'method':'approximate', 'bins' : 100})
    val_plot.add_labels(axs[idx, 1], 'SDB Green vs In Situ', 'In Situ (m)', 'SDB Green (m)')
    val_plot.add_residuals(error, axs[idx, 2], metrics=['MedAE', 'Abs_std', 'MSD'], x_lim=10)
    axs[idx, 1].grid()
    axs[idx, 2].grid()
    
    # Añadir recuadro para cada fila
    rect = plt.Rectangle((rect_params[idx][0], rect_params[idx][1]), 
                         rect_params[idx][2], rect_params[idx][3], 
                         transform=fig.transFigure, 
                         fill=False, 
                         edgecolor='black', 
                         linewidth=2)
    fig.patches.append(rect)
    
    # Añadir nombre del método fuera del recuadro
    fig.text(text_params[idx][0], text_params[idx][1], name, 
             fontsize=18, 
             fontfamily='times new roman', 
             rotation=90, 
             va='center', 
             ha='center')

fig.tight_layout(rect=[0.05, 0.05, 0.95, 0.95])
# fig.savefig(r'C:\Users\sergi\Documents\repos\gee_acolite\figs\sdb.png', dpi = 300, bbox_inches = 'tight')